In [ ]:
import matplotlib.pyplot as plt

from link_prediction.config import (
    RESULTS_DIR,
    load_experiment_config,
    load_methods_config,
)
from link_prediction.robustness import (
    run_negative_sampling_robustness,
)

In [ ]:
benchmark_name = "standard"

experiment_config = (
    load_experiment_config()
)

methods_config = (
    load_methods_config()
)

confirmatory_method_ids = list(
    experiment_config[
        "statistics"
    ][
        "confirmatory_methods"
    ]
)

confirmatory_method_names = {
    methods_config[
        "methods"
    ][
        method_id
    ][
        "name"
    ]
    for method_id
    in confirmatory_method_ids
}

figure_directory = (
    RESULTS_DIR
    / "figures"
    / benchmark_name
)

figure_directory.mkdir(
    parents=True,
    exist_ok=True,
)

(
    fold_metrics,
    network_summary,
    overall_summary,
) = run_negative_sampling_robustness(
    benchmark_name=
        benchmark_name,
    resume=True,
)

In [ ]:
overall_summary[
    [
        "negative_ratio",
        "is_primary_ratio",
        "family",
        "method_id",
        "method",
        "network_count",
        "average_precision_mean",
        "average_precision_sd_across_networks",
        "roc_auc_mean",
        "roc_auc_sd_across_networks",
        "average_precision_rank",
    ]
].sort_values(
    [
        "negative_ratio",
        "average_precision_rank",
        "method",
    ]
).reset_index(
    drop=True
)

In [ ]:
performance_matrix = (
    overall_summary
    .pivot(
        index="method",
        columns="negative_ratio",
        values="average_precision_mean",
    )
)

primary_ranks = (
    overall_summary[
        overall_summary[
            "is_primary_ratio"
        ]
    ][
        [
            "method",
            "average_precision_rank",
        ]
    ]
    .set_index(
        "method"
    )
)

performance_matrix = (
    performance_matrix
    .join(
        primary_ranks
    )
    .sort_values(
        [
            "average_precision_rank",
            "method",
        ]
    )
    .drop(
        columns=[
            "average_precision_rank",
        ]
    )
)

figure, axis = plt.subplots(
    figsize=(
        8,
        11,
    )
)

image = axis.imshow(
    performance_matrix.to_numpy(),
    aspect="auto",
    cmap="Blues",
    vmin=0.0,
    vmax=1.0,
)

axis.set_xticks(
    range(
        len(
            performance_matrix.columns
        )
    )
)

axis.set_xticklabels(
    [
        f"1:{ratio}"
        for ratio
        in performance_matrix.columns
    ]
)

axis.set_yticks(
    range(
        len(
            performance_matrix.index
        )
    )
)

axis.set_yticklabels(
    performance_matrix.index
)

axis.set_xlabel(
    "Positive-to-negative sampling ratio"
)

axis.set_ylabel(
    "Method"
)

colorbar = figure.colorbar(
    image,
    ax=axis,
)

colorbar.set_label(
    "Mean average precision"
)

figure.tight_layout()

figure.savefig(
    figure_directory
    / "negative_sampling_method_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure_directory
    / "negative_sampling_method_heatmap.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
confirmatory_summary = (
    overall_summary[
        overall_summary[
            "method_id"
        ].isin(
            confirmatory_method_ids
        )
    ]
)

figure, axis = plt.subplots(
    figsize=(
        8,
        5,
    )
)

for method_id in (
    confirmatory_method_ids
):
    method_summary = (
        confirmatory_summary[
            confirmatory_summary[
                "method_id"
            ]
            == method_id
        ]
        .sort_values(
            "negative_ratio"
        )
    )

    axis.errorbar(
        method_summary[
            "negative_ratio"
        ],
        method_summary[
            "average_precision_mean"
        ],
        yerr=method_summary[
            "average_precision_sd_across_networks"
        ],
        marker="o",
        capsize=4,
        label=method_summary[
            "method"
        ].iloc[0],
    )

axis.set_xticks(
    sorted(
        overall_summary[
            "negative_ratio"
        ].unique()
    )
)

axis.set_xticklabels(
    [
        f"1:{ratio}"
        for ratio
        in sorted(
            overall_summary[
                "negative_ratio"
            ].unique()
        )
    ]
)

axis.set_xlabel(
    "Positive-to-negative sampling ratio"
)

axis.set_ylabel(
    "Mean average precision"
)

axis.grid(
    alpha=0.25,
)

axis.legend()

figure.tight_layout()

figure.savefig(
    figure_directory
    / "confirmatory_negative_sampling_robustness.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure_directory
    / "confirmatory_negative_sampling_robustness.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
figure, axis = plt.subplots(
    figsize=(
        10,
        8,
    )
)

for method, method_summary in (
    overall_summary.groupby(
        "method",
        sort=False,
    )
):
    method_summary = (
        method_summary.sort_values(
            "negative_ratio"
        )
    )

    is_confirmatory = (
        method
        in confirmatory_method_names
    )

    axis.plot(
        method_summary[
            "negative_ratio"
        ],
        method_summary[
            "average_precision_rank"
        ],
        marker=(
            "o"
            if is_confirmatory
            else None
        ),
        linewidth=(
            2.5
            if is_confirmatory
            else 0.8
        ),
        alpha=(
            1.0
            if is_confirmatory
            else 0.3
        ),
        color=(
            None
            if is_confirmatory
            else "#808080"
        ),
        label=(
            method
            if is_confirmatory
            else None
        ),
    )

axis.set_xticks(
    sorted(
        overall_summary[
            "negative_ratio"
        ].unique()
    )
)

axis.set_xticklabels(
    [
        f"1:{ratio}"
        for ratio
        in sorted(
            overall_summary[
                "negative_ratio"
            ].unique()
        )
    ]
)

axis.set_xlabel(
    "Positive-to-negative sampling ratio"
)

axis.set_ylabel(
    "Average precision rank"
)

axis.invert_yaxis()

axis.grid(
    alpha=0.25,
)

axis.legend()

figure.tight_layout()

figure.savefig(
    figure_directory
    / "negative_sampling_rank_stability.png",
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    figure_directory
    / "negative_sampling_rank_stability.pdf",
    bbox_inches="tight",
)

plt.show()